# HIV/AIDS in California, 2010–2023: Dashboard

In [78]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
import ipywidgets as widgets
from ipywidgets import interact, interactive
from IPython.display import display, HTML

In [79]:
#Loading in data
#Check for missing values is in looking_at_data_options.ipynb


living_cases_url = 'https://virginia.box.com/shared/static/hfkttdmmyd4dg3sh7qumusevxxduimyq.csv'
living_cases = pd.read_csv(living_cases_url)
new_cases_url = 'https://virginia.box.com/shared/static/zjmyugj5gh68e5kuw2a923w1r2t55l0s.csv'
new_cases = pd.read_csv(new_cases_url)
deaths_url = 'https://virginia.box.com/shared/static/cssyj0b90kbzjesyfjfd08ohrownn0uj.csv'
deaths = pd.read_csv(deaths_url)

# Combine the CSVs without error
# Solution provided by Prof. Alvarado

'''It turns out that each table had two different category names, corresponding to their unit of observation.
So, [Prof. Alvarado] just changed them, but the user should know what 'Age' and 'City' actually mean when viewing results. We also added County'''

key = ['Year', 'Category', 'Group']

df1 = living_cases.rename(columns={'Count': 'Count_Living'})
df1['Category'] = df1['Category'].replace({'Age at Year End (in years)':'Age', 'City of Current Residence':'City', 'County of Current Residence':'County'})
df1 = df1.set_index(key)

df2 = new_cases.rename(columns={'Count': 'Count_New'})
df2['Category'] = df2['Category'].replace({'Age at Diagnosis (in years)':'Age', 'City of Residence at Diagnosis':'City', 'County of Residence at Diagnosis':'County'})
df2 = df2.set_index(key)

df3 = deaths.rename(columns={'Count': 'Count_Deaths'})
df3['Category'] = df3['Category'].replace({'Age at Death (in years)':'Age', 'City of Residence at Death':'City', 'County of Residence at Death':'County'})
df3 = df3.set_index(key)

hiv_aids = df1.join(df2).join(df3)

In [80]:
#Population totals per Year/Category/Group
pop_by_group_url = 'https://virginia.box.com/shared/static/k8o4d1qaewdguykqaz0iau6fv9arxyjy.csv'
pop_by_group = pd.read_csv(pop_by_group_url)

#back to regular columns so the keys are available for plotting and grouping
hiv_aids = hiv_aids.reset_index()

#left join so Current Gender and Transmission keep a null population rather than being dropped
hiv_aids = hiv_aids.merge(pop_by_group, on=['Year','Category','Group'], how='left')

In [ ]:
#theme + color palette
#one dark theme shared by every panel, layered on top of plotly's built-in dark template, with all text in white
#near-black instead of pure black is easier on the eyes, and only horizontal gridlines are kept, very faint
pio.templates['dashboard'] = go.layout.Template(layout={
    'paper_bgcolor': '#0E0E10', 'plot_bgcolor': '#0E0E10',
    'font': {'color': 'white'},
    'xaxis': {'showgrid': False},
    'yaxis': {'gridcolor': '#2A2A2A', 'zeroline': False}})
pio.templates.default = 'plotly_dark+dashboard'

#Voila's dark theme paints the page in JupyterLab's dark grays (#111111 around the edges, #212121 behind the cells), this sets both to the same color as the charts
display(HTML('<style>:root { --jp-layout-color0: #0E0E10; --jp-layout-color1: #0E0E10; }</style>'))

#amber and teal for the two groups the story is about, gray for every other group
highlight = {'Black/African American':'#F2A541', 'Latine':'#4FC3C9'}
muted = '#5A5A5F'
reference_gray = '#9A9A9A'

In [ ]:
#combining incidence + prevalence from leah and gaayathri's race breakdown into the first plot

#filtered to the 'Sex at Birth' category so no duplicates are accounted for
totals = hiv_aids[hiv_aids.Category == 'Sex at Birth'].groupby('Year')[['Count_New','Count_Living','Population']].sum().reset_index()
#calculated the incidence and prevalence rate
totals['Incidence']  = totals.Count_New    / totals.Population * 100_000
totals['Prevalence'] = totals.Count_Living / totals.Population * 100_000

#percent change since the first year, so incidence (around 10-14 per 100k) and prevalence (around 300-365 per 100k) can share one y-axis
totals['Incidence_Change']  = (totals.Incidence  / totals.Incidence.iloc[0]  - 1) * 100
totals['Prevalence_Change'] = (totals.Prevalence / totals.Prevalence.iloc[0] - 1) * 100

first, last = totals.Year.min(), totals.Year.max()

#new diagnosis rate for each race group
race = hiv_aids[hiv_aids.Category == 'Race/Ethnicity'].sort_values('Year')
race['Incidence'] = race.Count_New / race.Population * 100_000

#dropping groups too small for a stable rate
#unknown race is dropped too, since it has no population to divide by
race['Group_Size'] = race.groupby('Group').Count_Living.transform('mean')
race = race[(race.Group_Size >= 1000) & race.Population.notna()]
#multiple races is dropped as well, since its numbers likely reflect changes in how race is recorded rather than real trends
race = race[race.Group != 'Multiple races']

latest = race[race.Year == last].set_index('Group').Incidence

def plot_incidence(by_race):
    if not by_race:
        #default view: Leah's interactive line plot of incidence and prevalence
        #switched from two y-axes to one axis of percent change since the first year, so the two slopes can be compared honestly. hovering still shows the actual rate per 100k
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=totals.Year, y=totals.Incidence_Change, name="Incidence (new diagnoses)", mode="lines+markers", line={'color': "#ee19b2"}, customdata=totals.Incidence, hovertemplate="%{y:+.0f}% (%{customdata:.1f} per 100k)"))
        fig.add_trace(go.Scatter(x=totals.Year, y=totals.Prevalence_Change, name="Prevalence (living with HIV)", mode="lines+markers", line={'color': "#f7fbfb", 'dash': "dash"}, customdata=totals.Prevalence,hovertemplate="%{y:+.0f}% (%{customdata:.0f} per 100k)"))

        #a line at 0% marks the starting level
        fig.add_hline(y=0, line_color=reference_gray, line_width=1)

        #added a shaded region to highlight the covid-19 period
        fig.add_vrect(x0=2019.5, x1=2020.5, fillcolor="gray", opacity=0.15, line_width=0, annotation_text="COVID-19", annotation_position="top left")

        #configured and added context for this plot (width matches the race view so the row doesn't shift when switching, and leaves room for the dots next to it)
        fig.update_layout(title=f'HIV Incidence and Prevalence in California, {first}–{last}<br><span style="font-size:13px">Percent change in each rate per 100,000 Californians since {first}</span>', xaxis_title="Year", yaxis_title=f"% change since {first}", yaxis_ticksuffix="%", hovermode="x unified", legend={'orientation': "h", 'y': -0.2}, height=550, width=680)
        display(fig)
        return

    #closer look: new diagnoses broken down by race
    fig = go.Figure()
    for group, rows in race.groupby('Group'):
        color = highlight.get(group, muted)
        #labeling each line at its last point instead of using a legend (long names split onto two lines)
        fig.add_trace(go.Scatter(x=rows.Year, y=rows.Incidence, name=group, mode='lines+text', line={'color': color}, text=[''] * (len(rows) - 1) + [group.replace('/', '/<br>')], textposition='middle right', textfont={'color': color}, cliponaxis=False))
    title = (f'Black Californians are diagnosed at {latest["Black/African American"] / latest["White"]:.1f}x the rate<br>of White Californians, and the Latine rate is rising')

    #statewide line as a dashed gray reference behind the race lines
    fig.add_trace(go.Scatter(x=totals.Year, y=totals.Incidence, name='All Californians', mode='lines+text', line={'color': reference_gray, 'dash': 'dash'}, text=[''] * (len(totals) - 1) + ['All Californians'], textposition='middle right', cliponaxis=False))

    #added a shaded region to highlight the covid-19 period, when less testing likely lowered diagnoses
    fig.add_vrect(x0=2019.5, x1=2020.5, fillcolor="gray", opacity=0.15, line_width=0, annotation_text="COVID-19", annotation_position="top left")


    fig.update_layout(title=f'{title}<br><span style="font-size:13px">New HIV diagnoses per 100,000 Californians, {first}–{last}</span>', xaxis_title="Year", hovermode="x unified", showlegend=False, height=550, width=680, margin={'r': 130})
    #fixed y-axis limits (plotly's version of set_ylim): starting at 0 keeps the line heights proportional to the rates,
    #and 50 leaves room above the highest point (Black Californians, about 45 in 2010)
    fig.update_yaxes(range=[0, 50])
    display(fig)

In [83]:
#icon array: how many of every 100 Californians vs. of every 100 new diagnoses, belong to each race group in the latest year
latest_race = hiv_aids[(hiv_aids.Category == 'Race/Ethnicity') & (hiv_aids.Year == last)].set_index('Group')
#population share only uses groups with a population count, while the diagnosis share keeps unknown race in the total to reflect every new diagnosis
pop_share = latest_race.Population / latest_race.Population.sum() * 100
new_share = latest_race.Count_New / latest_race.Count_New.sum() * 100

#10x10 grid positions (bottom up)
dot_x = [i % 10 for i in range(100)]
dot_y = [i // 10 for i in range(100)]

def plot_dots(group):
    color = highlight.get(group, 'white')
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Californians', 'New diagnoses'))

    for col, share in enumerate([pop_share[group], new_share[group]], start=1):
        #one dot per percent
        dot_colors = [color if i < round(share) else "#2D2D36" for i in range(100)]
        fig.add_trace(go.Scatter(x=dot_x, y=dot_y, mode='markers', marker={'size': 14, 'color': dot_colors}), row=1, col=col)

    #hiding the axes
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    #plotly titles use <br> for line breaks
    fig.update_layout(title=f'{group} Californians are {pop_share[group]:.0f}% of the state and <br>{new_share[group]:.0f}% of new HIV diagnoses', showlegend=False, height=370, width=480, margin={'t': 150, 'l': 20, 'r': 20, 'b': 20})
    display(fig)

In [84]:
incidence_panel = interactive(plot_incidence, by_race=widgets.Checkbox(description="TAKE A CLOSER LOOK AT INCIDENCE", style={'description_width':'initial'}))
dots_panel = interactive(plot_dots, group=widgets.Dropdown(options=sorted(race.Group.unique()), value='Latine', description='Race'))

#each panel keeps a minimum width (its chart plus a little room), so it can't collapse while a chart is being redrawn, which otherwise lets plotly redraw the chart at the collapsed width and makes it tiny after toggling
incidence_panel.layout.min_width = '700px'
dots_panel.layout.min_width = '500px'

#centering the shorter dot panel vertically next to the line chart
widgets.HBox([incidence_panel, dots_panel], layout=widgets.Layout(align_items='center'))

In [ ]:
#American Indian/Alaska Native and Native Hawaiian/Pacific Islander are not included, since fewer than 1,000 people in each group are living with HIV in a typical year. The dot grid dropdown leaves them out too, but they are still counted when computing shares.

#Multiple races is left out of both panels because its numbers likely reflect changes in how race is recorded rather than useful data. '''

In [ ]:
# Californians living with HIV, newly diagnosed, or who died
# stacked area chart from katya rebuilt in plotly + interactivity


#each dropdown option points to its count column, and to how it reads in the subtitle
age_measures = {'Living with HIV':'Count_Living', 'Newly diagnosed':'Count_New', 'Died':'Count_Deaths'}
age_subtitles = {'Living with HIV':'living with HIV', 'Newly diagnosed':'newly diagnosed with HIV', 'Died':'with HIV who died'}

#keep only age data (the age bands already sort in order, from '0 to 11' up to '>= 75')
age_data = hiv_aids[hiv_aids.Category == 'Age']

#same viridis palette as the original chart, one color per age band, from the youngest to the oldest
age_colors = sample_colorscale('Viridis', len(age_data.Group.unique()))

@interact(measure=widgets.Dropdown(options=list(age_measures), description='Show'))
def plot_age(measure):
    #convert to wide format
    pivot = age_data.pivot(index='Year', columns='Group', values=age_measures[measure]).fillna(0)

    #convert each year to proportions, as percentages
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    fig = go.Figure()
    for group, color in zip(pivot_pct.columns, age_colors):
        #stackgroup stacks the bands like stackplot, with no outline so the bands blend straight into each other
        fig.add_trace(go.Scatter(x=pivot_pct.index, y=pivot_pct[group], name=group, stackgroup='age', line={'width': 0}, fillcolor=color, hovertemplate='%{y:.1f}%'))

    #titles
    titles = {'Living with HIV': 'An Aging Population with HIV', 'Newly diagnosed': 'Californians are Still Diagnosed at Similar Ages', 'Died': 'The Majority of Deaths of People Living with HIV are Among Older Californians'}

    #legend reversed so it reads in the same order as the stack, oldest at the top
    fig.update_layout(title=f'{titles[measure]}<br><span style="font-size:13px">Age distribution of Californians {age_subtitles[measure]}, {first}–{last}</span>', xaxis_title="Year", yaxis_title="Proportion of Population Living with HIV/AIDS)", yaxis_ticksuffix="%", hovermode="x unified", legend={'title': 'Age Group', 'traceorder': 'reversed'}, height=550, width=1160)
    display(fig)

interactive(children=(Dropdown(description='Show', options=('Living with HIV', 'Newly diagnosed', 'Died'), val…

In [ ]:
#sex and transmission panel
#gender from first interactive line plot + transmission fatality and grouped transmission categories to compare

#death rate among people living with HIV for cisgender women and men (per 1,000 living with HIV)
#current gender is used
gender_colors = {'Cisgender women':'#C3A6FF', 'Cisgender men':'#8FD694'}
gender = hiv_aids[hiv_aids.Category == 'Current Gender'].sort_values('Year')
gender['Death_Rate'] = gender.Count_Deaths / gender.Count_Living * 1000

#takeaway, calculated from the data: cisgender men's share of everyone living with HIV in the latest year
latest_gender = gender[gender.Year == last].set_index('Group').Count_Living
men_share = latest_gender['Cisgender men'] / latest_gender.sum() * 100

#transmission bar chart, switched from total deaths to deaths per 1,000 living with HIV since the totals mostly reflect how many people are in each group rather than how deadly each group's HIV is
#compare HIV/AIDS deaths by transmission risk group across all transmission categories
transmission = hiv_aids[hiv_aids['Category'].str.startswith('Transmission Category:', na=False)]

transmission = (transmission.groupby('Group', as_index=False)[['Count_Living', 'Count_Deaths']].sum())
transmission['Deaths_per_1000_living'] = transmission['Count_Deaths'] / transmission['Count_Living'] * 1000

#dropping groups with fewer than 1,000 people living with HIV in a typical year
transmission = transmission[transmission.Count_Living / len(totals) >= 1000]

#highest rate at top
transmission = transmission.sort_values('Deaths_per_1000_living')

#short labels
method_labels = {'Male-to-male sexual contact (MMSC)':'Male-to-male', 'MMSC & IDU':'Male-to-male + drug use',
                 'Injection drug use (IDU)':'Injection drug use', 'High-risk heterosexual contact (HRH)':'High-risk heterosexual',
                 'Heterosexual contact (Non-HRH)':'Other heterosexual', 'Sexual contact':'Sexual contact', 'Unknown risk':'Unknown'}
transmission['Label'] = transmission.Group.map(method_labels)

#share of cisgender women and men living with HIV by transmission method, all years combined like the death rates
cis_categories = {'Transmission Category: Cisgender Women Adult or Adolescent':'Cisgender women',
                  'Transmission Category: Cisgender Men Adult or Adolescent':'Cisgender men'}
cis_transmission = hiv_aids[hiv_aids.Category.isin(cis_categories)].copy()
cis_transmission['Gender'] = cis_transmission.Category.map(cis_categories)
cis_share = cis_transmission.groupby(['Gender','Group']).Count_Living.sum()
cis_share = (cis_share / cis_share.groupby(level='Gender').transform('sum') * 100).reset_index(name='Share')

#keeping the same methods (and labels) as the death rate bars so the two panels line up row by row
cis_share = cis_share[cis_share.Group.isin(transmission.Group)]
cis_share['Label'] = cis_share.Group.map(method_labels)

In [87]:
#three panels: gender on left and the two bar charts on the right
#the line needs to get the wider column
fig = make_subplots(rows=2, cols=2, specs=[[{'rowspan':2}, {}], [None, {}]], column_widths=[0.55, 0.45], horizontal_spacing=0.07, vertical_spacing=0.15, subplot_titles=('Deaths per 1,000 living with HIV, by gender','Injection drug use is the deadliest transmission method<br><span style="font-size:12px">Deaths per 1,000 living with HIV</span>', 'More women were infected through injection drug use<br><span style="font-size:12px">Share of each group living with HIV</span>'))

#aligning all of the subplot titles to the far edges and moving the left title up
fig.layout.annotations[0].update(x=0, xanchor='left', align='left', xshift=-22, yshift=12)
for panel_title in fig.layout.annotations[1:3]:
    panel_title.update(x=1, xanchor='right', align='right')

#left: one line per gender, named in the legend
#legendgroup ties each line to its gender's bars below, so one legend entry covers both (and clicking it hides both)
for group, color in gender_colors.items():
    rows = gender[gender.Group == group]
    fig.add_trace(go.Scatter(x=rows.Year, y=rows.Death_Rate, name=group, mode='lines', line={'color': color}, legendgroup=group, hovertemplate='%{y:.1f} per 1,000'), row=1, col=1)

#adding shaded region to highlight the covid-19 period
fig.add_vrect(x0=2019.5, x1=2020.5, fillcolor="gray", opacity=0.15, line_width=0, annotation_text="COVID-19", annotation_position="top left", row=1, col=1)

#color-coding the groups that involve injection drug use
bar_colors = ['white' if 'IDU' in group else muted for group in transmission.Group]
fig.add_trace(go.Bar(x=transmission.Deaths_per_1000_living, y=transmission.Label, orientation='h', marker_color=bar_colors, text=transmission.Deaths_per_1000_living, texttemplate='%{text:.1f}', textposition='outside', cliponaxis=False, customdata=transmission.Count_Deaths, hovertemplate='%{x:.1f} per 1,000 (%{customdata:,} deaths)<extra></extra>', showlegend=False), row=1, col=2)

#grouped bars in each gender's color, with the injection drug use bars at full opacity
for group, color in gender_colors.items():
    rows = cis_share[cis_share.Gender == group]
    fig.add_trace(go.Bar(x=rows.Share, y=rows.Label, orientation='h', name=group, marker={'color': color, 'opacity': [1 if 'IDU' in method else 0.45 for method in rows.Group]}, text=rows.Share, texttemplate='%{text:.0f}%', textposition='outside', cliponaxis=False, legendgroup=group, showlegend=False, hovertemplate='%{x:.1f}%<extra>%{fullData.name}</extra>'), row=2, col=2)

#both bar panels list the methods in the same order, and every bar is labeled. x-axes and gridlines are hidden
#flipping the bars

for row in (1, 2):
    fig.update_xaxes(visible=False, autorange='reversed', row=row, col=2)
    fig.update_yaxes(categoryorder='array', categoryarray=list(transmission.Label), showgrid=False, side='right', row=row, col=2)

fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_yaxes(range=[0, 20], dtick=2, row=1, col=1)

fig.update_layout(title=f'Women with HIV died at a higher rate than men in most years, even though men are {men_share:.0f}% of Californians living with HIV' f'<br><span style="font-size:13px">The gap mostly reflects how people were infected: injection drug use is the deadliest route,and far more women were infected through it. </span>', barmode='group', legend={'orientation': 'h', 'x': 0, 'y': -0.08}, height=750, width=1160, margin={'t': 180, 'r': 170})
display(fig)